## 🥈 Silver Layer

In the Silver Layer, we will perform data cleaning and transformation to prepare the data for analysis. This includes handling missing values, converting data types, and creating new features as needed.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
NAME_TABLE_SILVER = "sandbox_prd.silver_layer.streaming_history_user_spotify"

In [0]:
# Read the bronze table
df_bronze = spark.read.table("sandbox_prd.bronze_layer.streaming_history_user_spotify")

In [0]:
# List of columns to drop from the bronze layer that are not needed for analysis
drop_columns = [
    "audiobook_chapter_title",
    "audiobook_chapter_uri",
    "audiobook_title",
    "audiobook_uri",
    "conn_country",
    "incognito_mode",
    "ip_addr",
    "dt_ingestion",
    "source_file"
]

# Remove unnecessary columns to create the silver layer dataframe
df_silver = df_bronze.drop(*drop_columns)

In [0]:
# Create a new column for the hour of the day in Brazil timezone
v_hora_brasil = hour(from_utc_timestamp(col("ts"), "America/Sao_Paulo"))

In [0]:
df_silver = (
    df_silver
    .withColumnRenamed("episode_name",                      "nm_episode_name")
    .withColumnRenamed("episode_show_name",                 "nm_episode_show_name")
    .withColumnRenamed("master_metadata_album_album_name",  "nm_album_name")
    .withColumnRenamed("master_metadata_album_artist_name", "nm_artist_name")
    .withColumnRenamed("master_metadata_track_name",        "nm_track_name")
    .withColumnRenamed("ms_played",                         "qt_played_ms")
    .withColumnRenamed("offline",                           "fl_offline")
    .withColumnRenamed("offline_timestamp",                 "ts_offline")
    .withColumnRenamed("platform",                          "ds_platform")
    .withColumnRenamed("reason_end",                        "ds_reason_end")
    .withColumnRenamed("reason_start",                      "ds_reason_start")
    .withColumnRenamed("shuffle",                           "fl_shuffle")
    .withColumnRenamed("skipped",                           "fl_skipped")
    .withColumnRenamed("spotify_episode_uri",               "ds_spotify_episode_uri")
    .withColumnRenamed("spotify_track_uri",                 "ds_spotify_track_uri")
    .withColumn("ts_streaming",  col("ts").cast("Timestamp"))
    .withColumn("dt_referencia", to_date(col("ts")))
)

In [0]:
df_silver = (
    df_silver
    .withColumn("nr_hora_brasil",v_hora_brasil)
    .withColumn("dt_ano_mes",
                concat(
                        year(col("ts")).cast("String"),
                        lit("-"),
                        month(col("ts")).cast("String")
                        )
                )
    .withColumn("ts_duration_seconds", (col("qt_played_ms") / 1000).cast("Int"))
    .withColumn("ts_duration_minutes", (col("qt_played_ms") / 1000 / 60).cast("Int"))
    .withColumn("ds_day_of_week",      when(dayofweek(col("ts")) == 1, "Domingo")
                                       .when(dayofweek(col("ts")) == 2, "Segunda-feira")
                                       .when(dayofweek(col("ts")) == 3, "Terça-feira")
                                       .when(dayofweek(col("ts")) == 4, "Quarta-feira")
                                       .when(dayofweek(col("ts")) == 5, "Quinta-feira")
                                       .when(dayofweek(col("ts")) == 6, "Sexta-feira")
                                       .when(dayofweek(col("ts")) == 7, "Sábado"))
    .withColumn("ds_periodo_dia",      when(v_hora_brasil < 6,  "Madrugada")
                                       .when(v_hora_brasil < 12, "Manhã")
                                       .when(v_hora_brasil < 18, "Tarde")
                                       .otherwise("Noite"))
    .withColumn("nr_ordem_periodo",    when(v_hora_brasil < 6,  1)
                                       .when(v_hora_brasil < 12, 2)
                                       .when(v_hora_brasil < 18, 3)
                                       .otherwise(4))
)

In [0]:
df_silver = (
    df_silver
    .withColumn("ds_tipo_inicio", when(col("ds_reason_start") == "trackdone",            "Reprodução Automática")
                                  .when(col("ds_reason_start") == "clickrow",             "Seleção Manual")
                                  .when(col("ds_reason_start") == "appload",              "Retomada App")
                                  .when(col("ds_reason_start") == "playbtn",              "Botão Play")
                                  .when(col("ds_reason_start").isin("fwdbtn", "backbtn"), "Navegação (Pular/Voltar)")
                                  .when(col("ds_reason_start") == "remote",               "Controle Externo")
                                  .otherwise("Outros"))
    .withColumn("ds_device_type", when(lower(col("ds_platform")).contains("android"),     "Android")
                                  .when(lower(col("ds_platform")).contains("ios"),         "iOS")
                                  .when(lower(col("ds_platform")).contains("web"),         "Web")
                                  .when(lower(col("ds_platform")).contains("windows"),     "Windows")
                                  .when(lower(col("ds_platform")).contains("mac"),         "Mac")
                                  .when(lower(col("ds_platform")).contains("linux"),       "Linux")
                                  .when(lower(col("ds_platform")).contains("tv"),          "TV")
                                  .when(lower(col("ds_platform")).contains("echo_show_5"), "Echo_Show")
                                  .when(lower(col("ds_platform")).contains("other"),       "Outros")
                                  .otherwise("Não identificado"))
)
 

In [0]:
df_silver = (
    df_silver
    .withColumn("ds_link_musica", when(col("nm_track_name").isNotNull(),
                                       concat(lit("https://open.spotify.com/track/"),   col("ds_spotify_track_uri")))
                                  .when(col("nm_album_name").isNotNull(),
                                       concat(lit("https://open.spotify.com/album/"),   col("ds_spotify_track_uri")))
                                  .when(col("nm_episode_show_name").isNotNull(),
                                       concat(lit("https://open.spotify.com/episode/"), col("ds_spotify_track_uri")))
                                  .otherwise(lit("Não identificado")))
)
 

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "nm_audio_name",
        when(
            col("nm_episode_name").isNull(),
            col("nm_track_name")
        )
        .when(
            col("nm_track_name").isNull(),
            col("nm_episode_name")
        )
    )
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "nm_audio_type",
        when(
            col("nm_episode_name").isNull(),
            "Music"
        )
        .when(
            col("nm_track_name").isNull(),
            "Podcast"
        )
    )
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "nm_owner_name",
        when(
            col("nm_artist_name").isNull(),
            col("nm_episode_show_name")
        )
        .when(
            col("nm_episode_show_name").isNull(),
            col("nm_artist_name")
        )
    )
)

In [0]:
# Save the dataframe in Silver Layer
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(NAME_TABLE_SILVER)

print(f"✅ Carga concluída com sucesso em: {NAME_TABLE_SILVER}")